> **Historical analysis:** This notebook includes a Brand-to-J&J normalization that has since been withdrawn. Its saved results describe that earlier assumption, not the current cleaned dataset. Do not use those results as current manufacturer evidence. See [current decisions](../docs/decisions.md) and revalidate affected findings before reuse.

# What does one row represent, and what should be unique?

Audit the grain of the supplied dataset before choosing cleaning or deduplication rules. Test uniqueness at several levels: raw row, internal record key, retailer URL/listing, retailer SKU, plausible GTIN/product, and normalized product name. Repetition at one level can be legitimate at another.

**Scope:** observable evidence from these two exports, not a confirmed source schema. Read source CSVs as strings, preserve provenance, and apply no deletions or merges. Separate raw data from a provisional structural-screen view. Training labels and target-label absence are not part of product identity.


## Findings: uniqueness depends on what a row is meant to represent

The export has **112,634 source rows**; **111,992** pass the provisional structural screen. No rows were deleted or merged.

| Candidate identity | Repeated groups in screened combined data | Groups shared by training and target |
|---|---:|---:|
| JoiningKey | 0 | 0 |
| MDM_Id | 0 | 0 |
| retailer + exact URL | 0 | 0 |
| retailer + SKU | 807 | 333 |
| checksum-valid GTIN | 192 | 81 |
| retailer + normalized name | 1,624 | 689 |

**Conclusion:** this is not unique at every product/entity level. Preserve listing/source records while investigating candidate product groups. Uniqueness of a row identifier does not make SKU, GTIN, product name, labels, or timestamps unique. The audit below separates exact duplicates, structural errors, legitimate repeated attributes, candidate repeated products, and annotation conflicts.


In [1]:
from pathlib import Path
from urllib.parse import urlsplit,urlunsplit
import pandas as pd
import numpy as np
import hashlib,re,runpy
ROOT=Path.cwd()
if not (ROOT/'data'/'provided').exists():ROOT=ROOT.parent
frames=[]
for dataset in ['Training','Target']:
    frame=pd.read_csv(ROOT/'data'/'provided'/f'Selfcare_{dataset}_data (1).csv',dtype='string',low_memory=False)
    frame['_dataset']=dataset;frame['_source_row']=frame.index
    frames.append(frame)
data=pd.concat(frames,ignore_index=True)
SOURCE_COLUMNS=[c for c in data.columns if not c.startswith('_')]
TARGETS=['Mnfr','Brand','Platform','Segment','Sub-Segment','TargetAgeGroup']
def clean(s):return s.astype('string').str.strip().replace('',pd.NA)
for col,key in [('JoiningKey','_joining'),('MDM_Id','_id'),('Retailer','_retailer'),('ProductUrl','_url'),
                ('Sku','_sku'),('Upc','_upc'),('ProductName','_name'),('MDM_InsertDateTime','_time')]:data[key]=clean(data[col])
data['_retailer']=data['_retailer'].str.casefold()
data['_name']=data['_name'].str.casefold().str.replace(r'\s+',' ',regex=True)
data['_joining_hex']=data['_joining'].str.fullmatch(r'[0-9a-fA-F]{32}',na=False)
data['_id_integer']=data['_id'].str.fullmatch(r'\d+',na=False)
data['_url_http']=data['_url'].str.match(r'^https?://',case=False,na=False)
data['_time_plausible']=pd.to_numeric(data['_time'],errors='coerce').between(40000,60000).fillna(False)
# Preserve comparability with notebook 07; do not require JoiningKey format to select rows.
data['_screen']=data['_id_integer']&data['_url_http']&data['_time_plausible']
view=data.loc[data['_screen']].copy()
def url_base(v):
    if pd.isna(v):return pd.NA
    try:
        u=urlsplit(v);return urlunsplit((u.scheme.lower(),u.netloc.lower(),u.path,'',''))
    except ValueError:return pd.NA
view['_url_base']=view['_url'].map(url_base)
def gtin(v):
    if pd.isna(v) or not re.fullmatch(r'(?:\d{8}|\d{12}|\d{13}|\d{14})',v) or len(set(v))==1:return pd.NA
    total=sum(int(d)*(3 if i%2==0 else 1) for i,d in enumerate(reversed(v[:-1])))
    return v.zfill(14) if (10-total%10)%10==int(v[-1]) else pd.NA
view['_gtin']=view['_upc'].map(gtin).astype('string')
print('Raw rows:',len(data),'screened rows:',len(view),'flagged for review:',len(data)-len(view))
print(data.groupby('_dataset')['_screen'].agg(rows='size',screened='sum').to_string())


Raw rows: 112634 screened rows: 111992 flagged for review: 642
           rows  screened
_dataset                 
Target    28082     27909
Training  84552     84083


## 1. Column uniqueness is not row uniqueness

For every field, report completeness and distinct values. Classification labels, retailer names, ratings, categories, and timestamps are expected to repeat; enforcing uniqueness on them would destroy valid observations. These profiles use trimmed nonblank values but do not change the source columns.


In [2]:
profiles=[]
for dataset,frame in data.groupby('_dataset'):
    for col in SOURCE_COLUMNS:
        s=clean(frame[col]);counts=s.value_counts()
        profiles.append({'dataset':dataset,'column':col,'rows':len(frame),'nonblank':int(s.notna().sum()),
         'distinct':s.nunique(),'repeated_values':int(counts.gt(1).sum()),
         'rows_in_repeated_values':int(counts[counts.gt(1)].sum()),'unique_among_nonblank':not counts.gt(1).any()})
column_profile=pd.DataFrame(profiles)
for dataset in ['Training','Target']:
    print('\n'+dataset);print(column_profile.loc[column_profile.dataset.eq(dataset)].drop(columns='dataset').to_string(index=False))



Training
             column  rows  nonblank  distinct  repeated_values  rows_in_repeated_values  unique_among_nonblank
            Exclude 84552      5297         4                2                     5295                  False
         JoiningKey 84552     84552     84551                1                        2                  False
           Retailer 84552     84552         9                7                    84550                  False
           Category 84552     84551         3                2                    84550                  False
               Mnfr 84552     78854         4                3                    78853                  False
              Brand 84552     84230       146              141                    84225                  False
           Platform 84552      1105        63               56                     1098                  False
            Segment 84552     79763        10                9                    79762               

## 2. Exact duplicate rows and repeated product payloads

Count exact duplicates across all 32 source columns, excluding added provenance. Separately compare selected product-observation fields while ignoring labels, internal IDs, insertion time, and annotation fields. Identical payloads still need provenance review before merging; timestamps can represent separate observations. Rows differing only in whitespace are not counted as exact raw duplicates.


In [3]:
PAYLOAD=['Retailer','ProductCategory','ProductBrand','ProductName','ProductRating','ProductReviewsCount',
 'XRatXRev','ReviewsCount','Sku','Upc','ProductModelNumber','ProductUrl','ProductImageUrl','ProductDescription','ProductContents']
duplicate_rows=[]
for label,frame in [('raw',data),('screened',view)]:
    for scope,cols in [('all source columns',SOURCE_COLUMNS),('selected product payload',PAYLOAD)]:
        duplicate_rows.append({'view':label,'comparison':scope,'extra_duplicate_rows':int(frame.duplicated(cols).sum()),
                              'all_rows_in_duplicate_sets':int(frame.duplicated(cols,keep=False).sum())})
        examples=frame.loc[frame.duplicated(cols,keep=False)]
        if len(examples):
            print('\nExamples:',label,scope)
            print(examples[['_dataset','_source_row','JoiningKey','MDM_Id','Retailer','Sku','ProductName']].head(10).to_string(index=False,max_colwidth=90))
duplicates=pd.DataFrame(duplicate_rows);print(duplicates.to_string(index=False))



Examples: raw selected product payload
_dataset  _source_row                                                                                 JoiningKey MDM_Id                                                       Retailer  Sku                                                                                ProductName
Training        43276  which is found in our nails and hair* AMINO ACIDS AND VITAMIN C: Nourish your skin and...   <NA>  19 amino acids nourish the skins dermal layers; and Vitamin C <NA>                                                                                       <NA>
Training        70020  which is found in our nails and hair* AMINO ACIDS AND VITAMIN C: Nourish your skin and...   <NA>  19 amino acids nourish the skins dermal layers; and Vitamin C <NA> Serving Size 3 Tablets. Calories: 10; Protein: 3 g; Vitamin C (as calcium ascorbate): 3...
  Target         1700  which is found in our nails and hair* AMINO ACIDS AND VITAMIN C: Nourish your skin and...   <NA>  19

## 3. Candidate keys: global versus retailer-specific uniqueness

Exclude missing components from each key's uniqueness calculation and report how many rows are eligible. A unique key with missing values cannot identify every row. Check both exports together, not only each file separately. UPC/GTIN keys require intact supported-length digits and a valid check digit; scientific-notation values are not reconstructed. GTIN and normalized-name matches are candidate product matches, not verified equivalence.


In [4]:
KEYS={'JoiningKey':['_joining'],'MDM_Id':['_id'],'exact URL':['_url'],
 'retailer + exact URL':['_retailer','_url'],'retailer + URL base':['_retailer','_url_base'],
 'SKU globally':['_sku'],'retailer + SKU':['_retailer','_sku'],
 'checksum-valid GTIN':['_gtin'],'normalized name':['_name'],'retailer + normalized name':['_retailer','_name']}
key_tables={};key_rows=[]
for name,keys in KEYS.items():
    eligible=view.dropna(subset=keys)
    table=eligible.groupby(keys).agg(rows=('_id','size'),ids=('_id','nunique'),datasets=('_dataset','nunique'),
      retailers=('_retailer','nunique'),urls=('_url','nunique'),skus=('_sku','nunique'),names=('_name','nunique'),times=('_time','nunique'))
    key_tables[name]=table;repeated=table.rows.gt(1)
    key_rows.append({'key':name,'eligible_rows':len(eligible),'distinct_keys':len(table),
       'repeated_key_groups':int(repeated.sum()),'rows_in_repeated_groups':int(table.loc[repeated,'rows'].sum()),
       'cross_retailer_groups':int(table.retailers.gt(1).sum()),'cross_dataset_groups':int(table.datasets.gt(1).sum())})
key_summary=pd.DataFrame(key_rows);print(key_summary.to_string(index=False))
print('\nRaw versus screened internal-key collisions:')
for name,key in [('JoiningKey','_joining'),('MDM_Id','_id')]:
    for label,frame in [('raw',data),('screened',view)]:
        counts=frame[key].value_counts();print(name,label,'repeated values:',int(counts.gt(1).sum()),'missing:',int(frame[key].isna().sum()))
print('\nRepeated SKU groups by retailer:')
sku=view.dropna(subset=['_retailer','_sku']).groupby(['_retailer','_sku']).size()
print(sku.gt(1).groupby(level=0).agg(groups='size',repeated_groups='sum').to_string())


                       key  eligible_rows  distinct_keys  repeated_key_groups  rows_in_repeated_groups  cross_retailer_groups  cross_dataset_groups
                JoiningKey         111992         111992                    0                        0                      0                     0
                    MDM_Id         111992         111992                    0                        0                      0                     0
                 exact URL         111992         111992                    0                        0                      0                     0
      retailer + exact URL         111992         111992                    0                        0                      0                     0
       retailer + URL base         111992         111960                   32                       64                      0                     8
              SKU globally         111987         111125                  807                     1669          

## 4. What do JoiningKey and the source formatting tell us?

A 32-character hexadecimal value is hash-shaped, but shape alone does not prove the algorithm or input. Test a few explicit MD5 hypotheses (exact URL, lowercase URL, and MDM_Id). Partial agreement cannot establish a universal key-generation rule. Do not regenerate JoiningKey during cleaning without a confirmed specification.

The screen uses a plausible Excel-serial-like timestamp range, not a verified schema. Inspect failed rows rather than silently deleting them. Unique screened IDs must not hide raw export problems.


In [5]:
print('JoiningKey 32-hex format: raw',int(data['_joining_hex'].sum()),'of',len(data),
      '; screened',int(view['_joining_hex'].sum()),'of',len(view))
for label,values in [('MD5 exact trimmed URL',view['_url']),('MD5 lowercase URL',view['_url'].str.lower()),('MD5 MDM_Id',view['_id'])]:
    digest=values.map(lambda x:hashlib.md5(x.encode('utf-8')).hexdigest() if pd.notna(x) else None)
    match=digest.eq(view['_joining'].str.lower())
    print(label,':',int(match.sum()),'matches of',len(view))
print('\nFlagged source rows (representative first 12):')
print(data.loc[~data['_screen'],['_dataset','_source_row','JoiningKey','MDM_Id','MDM_InsertDateTime','Retailer','ProductUrl','ProductName']].head(12).to_string(index=False,max_colwidth=100))
print('\nAlternative screen sensitivity:')
for label,mask in [('integer ID + HTTP URL',data['_id_integer']&data['_url_http']),
                   ('integer ID + plausible time',data['_id_integer']&data['_time_plausible']),('all three',data['_screen'])]:
    print(label,'rows',int(mask.sum()),'repeated MDM_Id values',int(data.loc[mask,'_id'].value_counts().gt(1).sum()))


JoiningKey 32-hex format: raw 112628 of 112634 ; screened 111992 of 111992
MD5 exact trimmed URL : 4067 matches of 111992
MD5 lowercase URL : 11299 matches of 111992
MD5 MDM_Id : 0 matches of 111992

Flagged source rows (representative first 12):
_dataset  _source_row                       JoiningKey                                                                                               MDM_Id                                                                                   MDM_InsertDateTime Retailer   ProductUrl                                                                                          ProductName
Training         1012 06335e8441e6c8dd47dbac581a2a4f59 https://i5.walmartimages.com/asr/22fd0142-bc4f-494e-818b-fe58c4a63213.a93cac56c2b6fbf1e5ca374242f... https://www.walmart.com/ip/Entil-Large-Heating-Pad-24-x-33-for-Back-Shoulder-with-4-Heat-Levels-2...  Walmart 650380556316                                             "Entil Large Heating Pad 24"" x 33"" for Back & Sh

## 5. Repetition does not necessarily mean duplicate products

Inspect repeated retailer/SKU groups and cross-retailer GTIN groups alongside names, URLs, timestamps, model numbers, and UPCs. Differences may indicate package variants, separate listings, URL changes, repeated ingestion, or data errors. Identical product names are insufficient to decide between these explanations. Do not strip query parameters and automatically merge variants.


In [6]:
for key_name in ['retailer + SKU','checksum-valid GTIN','retailer + normalized name','retailer + URL base']:
    keys=KEYS[key_name];table=key_tables[key_name]
    selected=table.loc[table.rows.gt(1)].sort_values('rows',ascending=False).head(4).reset_index()[keys]
    examples=view.merge(selected,on=keys,how='inner')
    print('\nRepeated-group examples:',key_name)
    print(examples[['_dataset','_source_row','_id','Retailer','Sku','Upc','ProductModelNumber','_time','ProductName','ProductUrl']].head(30).to_string(index=False,max_colwidth=110))



Repeated-group examples: retailer + SKU
_dataset  _source_row     _id Retailer      Sku          Upc ProductModelNumber       _time                                                             ProductName                            ProductUrl
Training         5199 1136763   Target 79797486 036005009816               <NA> 45117.83254                   Yogasleep Dohm Classic White Noise Sound Machine, Tan https://www.target.com/p/-/A-79797495
Training         5984  373018   Target 79797486 036005009816               <NA>       44967                   Yogasleep Dohm Classic White Noise Sound Machine, Tan https://www.target.com/p/-/A-79797492
Training        10760  326210   Target 84637340  37000820628               <NA>       44936                              Puffs Ultra Soft Facial Tissue - 8pk/124ct https://www.target.com/p/-/A-79683997
Training        15324  324320   Target 11299923  41100820716               <NA>       44936 Miralax Laxative Powder for Gentle Constipation Relief 30 D

## 6. Are repeated candidate entities labeled consistently?

Compare nonblank target values only; missing versus populated is incompleteness, not a conflicting value. Use raw labels first, then an analysis copy with the user-confirmed J&J whitelist normalization and the documented Other Lifestyle ? Other Lifestyle CHC change. This preserves the distinction between source data and accepted corrections.

A disagreement within a candidate key is a review signal, not proof of an erroneous label: keys can merge variants, different intended ages, or pack forms. Do not majority-vote labels automatically.


In [7]:
whitelist=runpy.run_path(str(ROOT/'src'/'whitelist.py'))['jj_whitelist']
label_views={'raw labels':view.copy(),'accepted normalization':view.copy()}
for frame in label_views.values():
    for col in TARGETS:frame[col]=clean(frame[col])
normalized=label_views['accepted normalization']
normalized.loc[normalized['Brand'].isin(whitelist),'Mnfr']='J&J'
normalized.loc[normalized['Sub-Segment'].eq('Other Lifestyle'),'Sub-Segment']='Other Lifestyle CHC'
conflict_rows=[]
for version,frame in label_views.items():
    for key_name in ['retailer + SKU','checksum-valid GTIN','retailer + normalized name']:
        keys=KEYS[key_name];eligible=frame.dropna(subset=keys)
        grouped=eligible.groupby(keys)
        distinct=grouped[TARGETS].nunique(dropna=True)
        conflicts=distinct.gt(1)
        conflict_rows.append({'label_view':version,'key':key_name,'groups_with_any_conflict':int(conflicts.any(axis=1).sum()),
                             **{col:int(conflicts[col].sum()) for col in TARGETS}})
        if version=='accepted normalization':
            chosen=conflicts.loc[conflicts.any(axis=1)].head(3).reset_index()[keys]
            examples=eligible.merge(chosen,on=keys,how='inner')
            print('\nConflict examples after accepted normalization:',key_name)
            print(examples[['_dataset','_source_row','_id','Retailer','Sku','Upc','ProductName']+TARGETS].head(20).to_string(index=False,max_colwidth=90))
print('\nConflict counts:');print(pd.DataFrame(conflict_rows).to_string(index=False))



Conflict examples after accepted normalization: retailer + SKU
_dataset  _source_row     _id Retailer    Sku  Upc                                                            ProductName       Mnfr        Brand        Platform             Segment   Sub-Segment TargetAgeGroup
Training         8651  585794      CVS 154641 <NA>               TYLENOL Extra Strength Caplets With 500 Mg Acetaminophen        J&J      Tylenol    Simply Sleep Internal Analgesics    Sleep Aids          Adult
Training        35523 1140240      CVS 173270 <NA>     CVS Health 12HR Maximum Strength Mucus DM Extended Release Tablets All others   CVS Health            <NA>                CCFS      Cold/Flu          Adult
Training        42222 1142132      CVS 184297 <NA>              Children's Tylenol Pain + Fever Relief Medicine, 4 fl. OZ        J&J      Tylenol    Pain + Fever Internal Analgesics Internal Pain       Children
Training        67551 1264289      CVS 154641 <NA>                    Tylenol Extra Strength

## 7. Does unique MDM_Id mean train and target are product-independent?

No shared row key only proves separation at that key's level. Report direct overlap of retailer/SKU, checksum-valid GTIN, and normalized-name keys. These measures overlap with each other and should not be summed. Names and even product identifiers require review before being used for automatic joins or label transfer.


In [8]:
overlaps=[]
for key_name,keys in KEYS.items():
    eligible=view.dropna(subset=keys)
    shared=key_tables[key_name].loc[key_tables[key_name].datasets.gt(1)].reset_index()[keys]
    matched=eligible.merge(shared,on=keys,how='inner')
    overlaps.append({'key':key_name,'shared_groups':len(shared),
                     'training_rows_in_shared_groups':int(matched['_dataset'].eq('Training').sum()),
                     'target_rows_in_shared_groups':int(matched['_dataset'].eq('Target').sum())})
print(pd.DataFrame(overlaps).to_string(index=False))


                       key  shared_groups  training_rows_in_shared_groups  target_rows_in_shared_groups
                JoiningKey              0                               0                             0
                    MDM_Id              0                               0                             0
                 exact URL              0                               0                             0
      retailer + exact URL              0                               0                             0
       retailer + URL base              8                               8                             8
              SKU globally            333                             362                           341
            retailer + SKU            333                             362                           341
       checksum-valid GTIN             81                              86                            84
           normalized name           1047                       

## Cleaning decisions justified by this audit

| Observation | Cleaning implication |
|---|---|
| Classification labels, retailer, category, timestamps repeat | Preserve repeats; these are attributes, not primary keys. |
| Internal keys and exact listing URLs are unique in the screened export | Preserve them for traceability; do not infer canonical product identity or lifetime uniqueness. |
| Raw collisions occur on structurally suspicious rows | Quarantine for source-row review; repair only with evidence. The screen is provisional. |
| Same GTIN across retailers, different IDs | Retain listing records; attach a reviewed product-group mapping if needed. |
| Same retailer/SKU, different URLs/names/times | Review variants and source behavior before collapsing. Do not keep only the latest timestamp without an agreed entity definition. |
| Same name, different IDs | Candidate match only; check brand, strength, formulation, count, size, SKU/GTIN, and URLs. |
| Conflicting annotations within candidate groups | Review label definitions and grouping quality; preserve provenance and do not majority-vote automatically. |
| Products overlap across files under some keys | Do not assume independent products from unique MDM_Id. Build reviewed product groups for model evaluation. |

**Working interpretation:** the data is consistent with an export of retailer listing/source records, not a table guaranteed to contain one row per canonical product. Exact-URL uniqueness does not prove a persistent listing identifier, and a timestamp alone does not prove an ingestion event.

**What remains to confirm with the owner:** intended row grain (URL, SKU, variant, product, or observation), declared primary/unique keys, JoiningKey generation, MDM_Id persistence, meaning of insertion time, whether training/target were split by listing or product, and how age/pack/form variants should be grouped. Those answers determine whether any repeated observations should be merged. Until then, preserve the original rows and attach quality flags and candidate entity groups rather than deleting by a guessed key.
